In [1]:
import json
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd

import config
from src.db_io import leer_tabla_sqlite
from src.log_decisiones import registrar_decision

fact = leer_tabla_sqlite(config.ORO_DB, "fact_cliente_score")
df = leer_tabla_sqlite(config.ORO_DB, "cliente_features")
datos = fact.merge(df[["numero_id", "desc_segmento"]], on="numero_id", how="left")
(config.OUTPUTS_DIR / "powerbi").mkdir(parents=True, exist_ok=True)
(config.OUTPUTS_DIR / "eda").mkdir(parents=True, exist_ok=True)

por_nivel = (
    datos.groupby(["poblacion", "nivel"], as_index=False)
    .agg(n_clientes=("numero_id", "count"))
    .sort_values(["poblacion", "nivel"])
)
print("Clientes por nivel de prioridad y población:")
print(por_nivel.pivot(index="nivel", columns="poblacion", values="n_clientes").to_string())


Clientes por nivel de prioridad y población:
poblacion  con_historial  sin_historial
nivel                                  
A                 132368          82689
B                 132367          82688
C                 132368          82688
D                 132367          82688


In [2]:
# SPEC_V2 §7: monto potencial agregado por nivel, en los tres escenarios,
# SOLO para la población con historial. La población sin historial no tiene
# monto: su monto es NULL, no cero (§6.3).
con_hist = datos[datos["tiene_historial_inversion"] == 1]

montos = (
    con_hist.groupby("nivel", as_index=False)
    .agg(n_clientes=("numero_id", "count"),
         monto_conservador=("monto_conservador_12m", "sum"),
         monto_base=("monto_base_12m", "sum"),
         monto_optimista=("monto_optimista_12m", "sum"),
         # D5: descomposición además del total.
         monto_app_base=("monto_app_12m_base", "sum"),
         monto_prod_conservadores_base=("monto_prod_conservadores_12m_base", "sum"))
    .sort_values("nivel")
)
print("Monto potencial agregado a 12 meses (solo población con historial de inversión):")
print(montos.to_string(index=False))
print(f"\nClientes SIN historial de inversión (monto NULL, no cero): "
      f"{int((datos['tiene_historial_inversion'] == 0).sum()):,}")
print(
    "\nNOTA: estas sumas por nivel heredan la misma limitación estadística que "
    "el agregado total (ver el AVISO completo, con evidencia medida, en la "
    "celda del resumen ejecutivo más abajo): son la suma del límite p10/p90 de "
    "CADA cliente, no una banda de incertidumbre agregada válida."
)


Monto potencial agregado a 12 meses (solo población con historial de inversión):
nivel  n_clientes  monto_conservador    monto_base  monto_optimista  monto_app_base  monto_prod_conservadores_base
    A      132368      -2.327228e+11  1.859309e+12     3.065482e+12    8.754819e+11                   9.838267e+11
    B       42572      -6.728353e+11  7.433071e+05     3.879285e+11    0.000000e+00                   7.433071e+05
    C        4465      -7.056773e+10  7.795890e+04     4.068638e+10    0.000000e+00                   7.795890e+04
    D       40137      -1.394610e+12 -7.602586e+11    -3.945192e+11   -3.028551e+11                  -4.574034e+11

Clientes SIN historial de inversión (monto NULL, no cero): 640,681

NOTA: estas sumas por nivel heredan la misma limitación estadística que el agregado total (ver el AVISO completo, con evidencia medida, en la celda del resumen ejecutivo más abajo): son la suma del límite p10/p90 de CADA cliente, no una banda de incertidumbre agregada válida

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# BRUTO vs NETO: el brief de negocio y el modelo no miden lo mismo
# ─────────────────────────────────────────────────────────────────────────────
# La PRUEBA TÉCNICA pide "estimar el monto potencial que podrían invertir
# durante los próximos 12 meses" — un FLUJO DE ENTRADA. Lo que el modelo de
# monto proyecta (§6.3, notebooks/06_monto_12m.ipynb) es el CAMBIO NETO del
# saldo invertido, que sale negativo cuando el cliente viene retirando.
#
# Presentar solo el neto mezcla dos problemas de negocio distintos —captación y
# retención— y subestima el primero al restarle el segundo. Se separan aquí.
# Los negativos NO son un artefacto: en la ventana observada un cuarto de los
# clientes con historial redujo su saldo invertido; el modelo reproduce ese
# comportamiento, no lo inventa.
datos["monto_entrada_bruta"] = datos["monto_base_12m"].clip(lower=0)
datos["monto_salida_bruta"] = datos["monto_base_12m"].clip(upper=0)

con_hist = datos[datos["tiene_historial_inversion"] == 1]
flujos = {
    "n_clientes_entrada": int((con_hist["monto_base_12m"] > 0).sum()),
    "entrada_bruta_12m": float(con_hist["monto_entrada_bruta"].sum()),
    "n_clientes_salida": int((con_hist["monto_base_12m"] < 0).sum()),
    "salida_bruta_12m": float(con_hist["monto_salida_bruta"].sum()),
}
flujos["neto_12m"] = flujos["entrada_bruta_12m"] + flujos["salida_bruta_12m"]

print("DESCOMPOSICIÓN DEL MONTO BASE A 12 MESES (población con historial)")
print("-" * 78)
print(f"  Entrada bruta  (captación):  {flujos['entrada_bruta_12m']:>22,.0f} COP  "
      f"| {flujos['n_clientes_entrada']:>7,} clientes")
print(f"  Salida bruta   (retiro):     {flujos['salida_bruta_12m']:>22,.0f} COP  "
      f"| {flujos['n_clientes_salida']:>7,} clientes")
print(f"  {'-' * 74}")
print(f"  Neto:                        {flujos['neto_12m']:>22,.0f} COP")
print(
    "\nCUÁL CITAR, SEGÚN LA PREGUNTA:\n"
    "· '¿Cuánto podrían invertir?' (dimensionamiento del lanzamiento, que es lo "
    "que pide el brief) → ENTRADA BRUTA. Es el volumen que la App podría "
    "captar de los clientes que proyectan crecer.\n"
    "· '¿Cuánto crecerá el saldo invertido total?' (planeación financiera) → "
    "NETO. Ya descuenta los retiros proyectados.\n"
    "· La SALIDA BRUTA no es ruido ni un error de signo: es un segundo hallazgo "
    "con dueño distinto —retención, no captación— sobre una base de clientes "
    "identificable uno a uno."
)

# ─────────────────────────────────────────────────────────────────────────────
# BLOQUE COMERCIAL: el modelo de monto separa 3 regímenes, no 4 niveles
# ─────────────────────────────────────────────────────────────────────────────
# El 90.5% de los clientes tiene crecimiento real exactamente cero en los
# productos tipo App (ver notebooks/06_monto_12m.ipynb), así que el árbol
# colapsa a una predicción casi constante en la zona media y el recentrado por
# la mediana del error la deja en ~0. Consecuencia: los niveles intermedios no
# se diferencian entre sí por monto — presentarlos como dos prioridades
# distintas le atribuiría al modelo una resolución que no tiene.
#
# El bloque se DERIVA de la dispersión medida, no se codifica a mano: si un
# reentrenamiento futuro sí logra diferenciar la zona media, la reclasificación
# ocurre sola y nadie tiene que acordarse de actualizar una lista.
UMBRAL_DESV_SIN_SENAL = 1.0  # COP de desviación estándar intra-nivel

perfil_nivel = con_hist.groupby("nivel")["monto_base_12m"].agg(
    n="count", suma="sum", media="mean", desv="std", valores_distintos="nunique")
print("\nDispersión del monto base dentro de cada nivel:")
print(perfil_nivel.to_string())


def clasificar_bloque(nivel: str) -> str:
    """Régimen de monto de un nivel: sin señal, crecimiento o riesgo de retiro."""
    fila = perfil_nivel.loc[nivel]
    if fila["desv"] < UMBRAL_DESV_SIN_SENAL:
        return "sin_senal"
    return "crecimiento" if fila["media"] > 0 else "riesgo_retiro"


# La población sin historial queda fuera por construcción: no tiene monto (§6.3),
# y sus niveles se calcularon dentro de SU propia población, así que la etiqueta
# de bloque de un nivel con historial no aplica ahí.
datos["bloque_comercial"] = np.where(
    datos["tiene_historial_inversion"] == 1,
    datos["nivel"].map(clasificar_bloque),
    "sin_monto_estimable",
)
con_hist = datos[datos["tiene_historial_inversion"] == 1]

niveles_sin_senal = sorted(
    perfil_nivel.index[perfil_nivel["desv"] < UMBRAL_DESV_SIN_SENAL])
bloques = (
    con_hist.groupby("bloque_comercial", as_index=False)
    .agg(n_clientes=("numero_id", "count"),
         monto_base=("monto_base_12m", "sum"),
         monto_app_base=("monto_app_12m_base", "sum"),
         monto_prod_conservadores_base=("monto_prod_conservadores_12m_base", "sum"))
    .sort_values("monto_base", ascending=False)
)
print(f"\nNiveles sin señal de monto (desv. intra-nivel < {UMBRAL_DESV_SIN_SENAL} COP): "
      f"{niveles_sin_senal or 'ninguno'}")
print("\nAgregado por bloque comercial (población con historial):")
print(bloques.to_string(index=False))
print(
    "\nLECTURA: los niveles siguen siendo cuartiles válidos para PRIORIZAR el "
    "contacto comercial (se construyen sobre el score de propensión, que sí "
    "discrimina). Lo que este bloque advierte es distinto: para DIMENSIONAR el "
    "monto, los niveles intermedios no aportan resolución y deben presentarse "
    "juntos."
)

registrar_decision(
    clave="presentacion_bruto_neto_y_bloque_comercial",
    decision="reportar_entrada_bruta_salida_bruta_y_neto_por_separado_y_agrupar_niveles_sin_dispersion_de_monto",
    motivo=(
        "El brief pide 'el monto potencial que podrían invertir' (flujo de "
        "entrada) mientras que el modelo de §6.3 proyecta el cambio NETO del "
        f"saldo invertido. Medido en esta corrida: entrada bruta "
        f"{flujos['entrada_bruta_12m']:,.0f} COP sobre "
        f"{flujos['n_clientes_entrada']:,} clientes, salida bruta "
        f"{flujos['salida_bruta_12m']:,.0f} COP sobre "
        f"{flujos['n_clientes_salida']:,} clientes, neto "
        f"{flujos['neto_12m']:,.0f} COP. Reportar solo el neto subestima la "
        "oportunidad de captación y esconde una base de retención "
        "identificable. Adicionalmente, los niveles "
        f"{niveles_sin_senal} tienen dispersión de monto por debajo de "
        f"{UMBRAL_DESV_SIN_SENAL} COP (predicción constante tras el recentrado "
        "por mediana), así que no se diferencian entre sí para dimensionar y se "
        "agrupan en un bloque 'sin_senal'. La clasificación se deriva de la "
        "dispersión medida, no de una lista fija."
    ),
    evidencia={
        **flujos,
        "niveles_sin_senal": niveles_sin_senal,
        "umbral_desv_sin_senal": UMBRAL_DESV_SIN_SENAL,
        "desv_por_nivel": perfil_nivel["desv"].round(2).to_dict(),
    },
)


DESCOMPOSICIÓN DEL MONTO BASE A 12 MESES (población con historial)
------------------------------------------------------------------------------
  Entrada bruta  (captación):       1,859,309,351,232 COP  | 179,405 clientes
  Salida bruta   (retiro):           -760,258,559,891 COP  |  40,137 clientes
  --------------------------------------------------------------------------
  Neto:                             1,099,050,791,341 COP

CUÁL CITAR, SEGÚN LA PREGUNTA:
· '¿Cuánto podrían invertir?' (dimensionamiento del lanzamiento, que es lo que pide el brief) → ENTRADA BRUTA. Es el volumen que la App podría captar de los clientes que proyectan crecer.
· '¿Cuánto crecerá el saldo invertido total?' (planeación financiera) → NETO. Ya descuenta los retiros proyectados.
· La SALIDA BRUTA no es ruido ni un error de signo: es un segundo hallazgo con dueño distinto —retención, no captación— sobre una base de clientes identificable uno a uno.

Dispersión del monto base dentro de cada nivel:
      


Niveles sin señal de monto (desv. intra-nivel < 1.0 COP): ['B', 'C']

Agregado por bloque comercial (población con historial):
bloque_comercial  n_clientes    monto_base  monto_app_base  monto_prod_conservadores_base
     crecimiento      132368  1.859309e+12    8.754819e+11                   9.838267e+11
       sin_senal       47037  8.212660e+05    0.000000e+00                   8.212660e+05
   riesgo_retiro       40137 -7.602586e+11   -3.028551e+11                  -4.574034e+11

LECTURA: los niveles siguen siendo cuartiles válidos para PRIORIZAR el contacto comercial (se construyen sobre el score de propensión, que sí discrimina). Lo que este bloque advierte es distinto: para DIMENSIONAR el monto, los niveles intermedios no aportan resolución y deben presentarse juntos.


WindowsPath('C:/Users/natam/OneDrive/Desktop/Prueba-Tecnica-CREAN/outputs/decisiones/log_decisiones.csv')

In [4]:
por_segmento = (
    datos.groupby(["desc_segmento", "poblacion", "nivel"], as_index=False)
    .agg(n_clientes=("numero_id", "count"))
)
tabla_seg = por_segmento.pivot_table(
    index="desc_segmento", columns="nivel", values="n_clientes",
    aggfunc="sum", fill_value=0)
print("Distribución de niveles por desc_segmento:")
print(tabla_seg.to_string())


Distribución de niveles por desc_segmento:
nivel               A       B       C       D
desc_segmento                                
personal       100323  162231  197546  185870
plus           102838   50683   17342   25209
preferencial    11896    2141     168    3976


In [5]:
dimensionamiento = (
    datos.groupby(["nivel", "bloque_comercial", "poblacion", "desc_segmento"],
                  as_index=False)
    .agg(n_clientes=("numero_id", "count"),
         monto_conservador=("monto_conservador_12m", "sum"),
         monto_base=("monto_base_12m", "sum"),
         monto_optimista=("monto_optimista_12m", "sum"),
         # Bruto vs neto: `monto_base` es el neto; estas dos lo descomponen en
         # captación y retiro, que son problemas de negocio distintos.
         monto_entrada_bruta=("monto_entrada_bruta", "sum"),
         monto_salida_bruta=("monto_salida_bruta", "sum"),
         # D5: descomposición del monto base (además del total) por componente.
         monto_app_base=("monto_app_12m_base", "sum"),
         monto_prod_conservadores_base=("monto_prod_conservadores_12m_base", "sum"),
         score_medio=("score", "mean"))
    .sort_values(["nivel", "poblacion", "desc_segmento"])
)
dimensionamiento.to_csv(config.OUTPUTS_DIR / "powerbi" / "dimensionamiento.csv", index=False)

priorizados = datos[datos["nivel"].isin(["A", "B"])]
resumen = {
    "n_clientes_total": int(len(datos)),
    "n_clientes_priorizados_A_B": int(len(priorizados)),
    "n_nivel_A": int((datos["nivel"] == "A").sum()),
    "n_nivel_A_con_historial": int(((datos["nivel"] == "A") &
                                    (datos["poblacion"] == "con_historial")).sum()),
    "n_nivel_A_sin_historial": int(((datos["nivel"] == "A") &
                                    (datos["poblacion"] == "sin_historial")).sum()),
    # Titular del brief ("¿cuánto podrían invertir?"): la entrada bruta.
    "entrada_bruta_12m": flujos["entrada_bruta_12m"],
    "n_clientes_entrada": flujos["n_clientes_entrada"],
    "salida_bruta_12m": flujos["salida_bruta_12m"],
    "n_clientes_salida": flujos["n_clientes_salida"],
    "neto_12m": flujos["neto_12m"],
    "oportunidad_12m_conservador": float(con_hist["monto_conservador_12m"].sum()),
    "oportunidad_12m_base": float(con_hist["monto_base_12m"].sum()),
    "oportunidad_12m_optimista": float(con_hist["monto_optimista_12m"].sum()),
    "n_clientes_con_monto": int(con_hist["monto_base_12m"].notna().sum()),
    "niveles_sin_senal_de_monto": niveles_sin_senal,
}

# El agregado [conservador, optimista] de §7 se construye sumando el límite
# p10/p90 de CADA cliente por separado, lo que equivale a asumir que todos caen
# a la vez en su propio peor (o mejor) escenario. Se mide la evidencia en vez
# de asumirla; la celda siguiente construye el rango que sí se puede citar.
n_clientes_agregado = int(con_hist["monto_base_12m"].notna().sum())
ancho_medio_cliente = float(
    (con_hist["monto_optimista_12m"] - con_hist["monto_conservador_12m"]).mean())
ancho_agregado = resumen["oportunidad_12m_optimista"] - resumen["oportunidad_12m_conservador"]
factor_escala_medido = (
    ancho_agregado / ancho_medio_cliente if ancho_medio_cliente else float("nan"))
raiz_n = float(np.sqrt(n_clientes_agregado))
pct_banda_correlacion_perfecta = ancho_agregado / abs(resumen["oportunidad_12m_base"])
# Bajo independencia el ancho se reduce por el mismo factor sqrt(n)/n = 1/sqrt(n).
pct_banda_independencia = pct_banda_correlacion_perfecta / raiz_n

resumen["aviso_agregado_ingenuo"] = {
    "metodo": "suma_de_limites_p10_p90_por_cliente",
    "supuesto_implicito": "errores_de_backtest_perfectamente_correlacionados_entre_clientes",
    "n_clientes": n_clientes_agregado,
    "raiz_n": raiz_n,
    "factor_escala_medido_ancho_agregado_sobre_ancho_medio_cliente": factor_escala_medido,
    "ancho_relativo_correlacion_perfecta": pct_banda_correlacion_perfecta,
    "ancho_relativo_independencia": pct_banda_independencia,
    "recomendacion": (
        "no_citar_el_rango_agregado; "
        "usar_los_escenarios_de_captura_de_la_celda_siguiente"
    ),
}

print("=" * 78)
print("RESUMEN EJECUTIVO")
print("=" * 78)
print(f"Clientes totales scoreados:            {resumen['n_clientes_total']:,}")
print(f"Clientes priorizados (niveles A y B):  {resumen['n_clientes_priorizados_A_B']:,}")
print(f"  · nivel A con historial:             {resumen['n_nivel_A_con_historial']:,}")
print(f"  · nivel A sin historial (lookalike): {resumen['n_nivel_A_sin_historial']:,}")

print(f"\nOportunidad a 12 meses (población con historial, "
      f"{resumen['n_clientes_con_monto']:,} clientes):")
print(f"  entrada bruta — captación:  {resumen['entrada_bruta_12m']:>22,.0f} COP  "
      f"| {resumen['n_clientes_entrada']:>7,} clientes   ← titular del brief")
print(f"  salida bruta  — retención:  {resumen['salida_bruta_12m']:>22,.0f} COP  "
      f"| {resumen['n_clientes_salida']:>7,} clientes")
print(f"  neto (entrada + salida):    {resumen['neto_12m']:>22,.0f} COP")
print(
    "\nADVERTENCIAS DE INTERPRETACIÓN:\n"
    "· El modelo proyecta el CAMBIO NETO del saldo invertido, no un flujo de "
    "entrada. El brief pide 'el monto potencial que podrían invertir': esa "
    "pregunta la responde la ENTRADA BRUTA. El neto ya le restó los retiros "
    "proyectados y por lo tanto subestima la oportunidad de captación.\n"
    f"· La salida bruta corresponde a {resumen['n_clientes_salida']:,} clientes "
    "que el modelo proyecta desinvirtiendo. Es una base de RETENCIÓN accionable "
    "y nominada, no un error de signo del modelo.\n"
    f"· Los niveles {resumen['niveles_sin_senal_de_monto']} no tienen dispersión "
    "de monto (predicción constante): sirven para priorizar contacto, no para "
    "dimensionar. Presentarlos juntos como un solo bloque.\n"
    "· Horizonte de 12 meses extrapolado desde ~13 meses de historia, validado "
    "solo contra 3 meses (§6.3). El escenario 'base' está recentrado por la "
    "MEDIANA del error de backtest (el modelo sobre-predice sistemáticamente): "
    "es la versión YA corregida por sesgo, no la predicción cruda.\n"
    "· Los clientes 'A' sin historial se rankean por SIMILITUD (lookalike), no por "
    "probabilidad validada: en ese segmento la etiqueta es 0 por construcción (§6.1).\n"
    "· Los niveles NO son comparables entre poblaciones: cada 'A' es el 25% superior "
    "de SU población (§6.2)."
)

print(
    "\n" + "=" * 78 + "\n"
    "POR QUÉ EL RANGO AGREGADO [CONSERVADOR, OPTIMISTA] NO SE PUEDE CITAR\n"
    + "=" * 78 + "\n"
    f"Se construye sumando el límite p10/p90 de CADA cliente por separado "
    f"(n={n_clientes_agregado:,}), lo que equivale a asumir que los "
    f"{n_clientes_agregado:,} clientes caen SIMULTÁNEAMENTE en su propio "
    "escenario más pesimista (o más optimista): ignora por completo la "
    "diversificación entre clientes.\n"
    f"Evidencia medida: ancho del agregado / ancho medio por cliente = "
    f"{factor_escala_medido:,.1f}, es decir exactamente n. Eso da una banda de "
    f"{pct_banda_correlacion_perfecta:.0%} de la base "
    f"({resumen['oportunidad_12m_conservador']:,.0f} a "
    f"{resumen['oportunidad_12m_optimista']:,.0f} COP), que es la razón de que "
    "el extremo conservador salga negativo. Los escenarios POR CLIENTE sí son "
    "correctos y sirven para ordenar; es solo la AGREGACIÓN la que está mal.\n"
    f"Y el extremo opuesto tampoco sirve: bajo independencia la banda escalaría "
    f"con sqrt(n) = {raiz_n:,.0f} en vez de n = {n_clientes_agregado:,}, lo que "
    f"da un ancho de apenas {pct_banda_independencia:.1%}. Nadie cree que una proyección a "
    "12 meses hecha sobre ~13 meses de historia tenga esa precisión.\n"
    "Los dos extremos son absurdos y la correlación real de los errores entre "
    "clientes NO es estimable con una sola ventana temporal. Por eso el rango "
    "agregado no se deriva del error del modelo: se construye sobre una palanca "
    "de negocio explícita — celda siguiente."
)

RESUMEN EJECUTIVO
Clientes totales scoreados:            860,223
Clientes priorizados (niveles A y B):  430,112
  · nivel A con historial:             132,368
  · nivel A sin historial (lookalike): 82,689

Oportunidad a 12 meses (población con historial, 219,542 clientes):
  entrada bruta — captación:       1,859,309,351,232 COP  | 179,405 clientes   ← titular del brief
  salida bruta  — retención:        -760,258,559,891 COP  |  40,137 clientes
  neto (entrada + salida):         1,099,050,791,341 COP

ADVERTENCIAS DE INTERPRETACIÓN:
· El modelo proyecta el CAMBIO NETO del saldo invertido, no un flujo de entrada. El brief pide 'el monto potencial que podrían invertir': esa pregunta la responde la ENTRADA BRUTA. El neto ya le restó los retiros proyectados y por lo tanto subestima la oportunidad de captación.
· La salida bruta corresponde a 40,137 clientes que el modelo proyecta desinvirtiendo. Es una base de RETENCIÓN accionable y nominada, no un error de signo del modelo.
· Los niveles

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# EL RANGO QUE SÍ SE PUEDE CITAR: escenarios de captura comercial
# ─────────────────────────────────────────────────────────────────────────────
# La celda anterior deja acotado el problema: la banda estadística agregada es
# ~5x la base con correlación perfecta y ~1% con independencia, y la correlación
# real no se puede estimar con una sola ventana temporal.
#
# Ese callejón sin salida ES el hallazgo: el error del modelo no es la
# incertidumbre que manda. La que manda es la ADOPCIÓN — cuántos de los
# clientes identificados efectivamente mueven recursos a la App. Un modelo de
# saldos no puede responder eso; es una hipótesis comercial.
#
# Así que el rango se construye sobre una palanca explícita y discutible:
#
#     oportunidad_12m = entrada_bruta × tasa_de_captura
#
# La tasa vive en config.TASAS_CAPTURA para que el notebook y la interfaz
# (app/tablero.py) no puedan discrepar. Es un supuesto de negocio declarado,
# no una estimación disfrazada de precisión.
escenarios_captura = pd.DataFrame([
    {"escenario": nombre,
     "tasa_captura": tasa,
     "oportunidad_12m": resumen["entrada_bruta_12m"] * tasa}
    for nombre, tasa in config.TASAS_CAPTURA.items()
])

resumen["escenarios_captura"] = {
    "formula": "entrada_bruta_12m * tasa_de_captura",
    "base_de_calculo": resumen["entrada_bruta_12m"],
    "escenarios": {r.escenario: {"tasa_captura": r.tasa_captura,
                                 "oportunidad_12m": r.oportunidad_12m}
                   for r in escenarios_captura.itertuples()},
}

with open(config.OUTPUTS_DIR / "eda" / "resumen_ejecutivo.json", "w", encoding="utf-8") as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False)

print("=" * 78)
print("OPORTUNIDAD A 12 MESES POR ESCENARIO DE CAPTURA")
print("=" * 78)
print(f"Base de cálculo — entrada bruta: {resumen['entrada_bruta_12m']:,.0f} COP "
      f"({resumen['n_clientes_entrada']:,} clientes que proyectan crecer)\n")
for r in escenarios_captura.itertuples():
    print(f"  {r.escenario:<12} captura {r.tasa_captura:>4.0%}  →  "
          f"{r.oportunidad_12m:>20,.0f} COP")
print(
    "\nPOR QUÉ ESTE RANGO Y NO EL ESTADÍSTICO:\n"
    "· El supuesto queda explícito y es discutible con el negocio, que es "
    "exactamente lo que pide el brief ('definición explícita de los supuestos').\n"
    "· La tasa de captura es una palanca que el negocio controla; el error de "
    "backtest no lo es.\n"
    "· No finge una precisión que los datos no soportan.\n"
    "· Las tres cifras son positivas y del mismo orden de magnitud, así que se "
    "pueden presentar sin una nota al pie que las desmienta."
)

registrar_decision(
    clave="agregacion_rango_oportunidad_12m",
    decision="rango_agregado_por_escenarios_de_captura_comercial_no_por_error_del_modelo",
    motivo=(
        "El rango agregado NO se deriva del error del modelo. Medido en esta "
        f"corrida, los dos extremos estadísticos posibles son ambos inservibles: "
        f"sumar el p10/p90 de cada uno de los {n_clientes_agregado:,} clientes "
        f"asume correlación perfecta y da un ancho de {pct_banda_correlacion_perfecta:.0%} "
        "sobre la base (por eso el conservador sale negativo), mientras que "
        f"asumir independencia y escalar por sqrt(n)={raiz_n:,.0f} da apenas "
        f"{pct_banda_independencia:.1%}, precisión que una proyección a 12 meses "
        "sobre ~13 meses de historia no puede tener. La correlación real de los "
        "errores entre clientes no es estimable con una sola ventana temporal, "
        "así que ninguna de las dos es defendible. El hallazgo de fondo es que "
        "la incertidumbre que manda no es el error del modelo sino la ADOPCIÓN. "
        "Por eso el rango se construye como entrada_bruta x tasa_de_captura, con "
        "la tasa en config.TASAS_CAPTURA: un supuesto de negocio declarado y "
        "discutible en vez de una precisión estadística falsa. Los escenarios "
        "POR CLIENTE (p10/p90) se conservan y siguen siendo válidos para ordenar "
        "y priorizar; lo que se descarta es su SUMA."
    ),
    evidencia={
        "n_clientes": n_clientes_agregado,
        "raiz_n": raiz_n,
        "ancho_relativo_correlacion_perfecta": pct_banda_correlacion_perfecta,
        "ancho_relativo_independencia": pct_banda_independencia,
        "entrada_bruta_12m": resumen["entrada_bruta_12m"],
        "tasas_captura": config.TASAS_CAPTURA,
        "oportunidad_por_escenario": {
            r.escenario: r.oportunidad_12m for r in escenarios_captura.itertuples()},
    },
)

OPORTUNIDAD A 12 MESES POR ESCENARIO DE CAPTURA
Base de cálculo — entrada bruta: 1,859,309,351,232 COP (179,405 clientes que proyectan crecer)

  conservador  captura  10%  →       185,930,935,123 COP
  base         captura  25%  →       464,827,337,808 COP
  optimista    captura  40%  →       743,723,740,493 COP

POR QUÉ ESTE RANGO Y NO EL ESTADÍSTICO:
· El supuesto queda explícito y es discutible con el negocio, que es exactamente lo que pide el brief ('definición explícita de los supuestos').
· La tasa de captura es una palanca que el negocio controla; el error de backtest no lo es.
· No finge una precisión que los datos no soportan.
· Las tres cifras son positivas y del mismo orden de magnitud, así que se pueden presentar sin una nota al pie que las desmienta.


WindowsPath('C:/Users/natam/OneDrive/Desktop/Prueba-Tecnica-CREAN/outputs/decisiones/log_decisiones.csv')